# Lab 7: Time Series Analysis with Bokeh - Melbourne Daily Minimum Temperatures

This notebook completes all requirements for Lab 7 using the Melbourne daily minimum temperatures dataset.

**Dataset**: `datasets/daily_min_temperatures.csv` (1981-1990 daily data)

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Bokeh imports
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import (ColumnDataSource, Whisker, FactorRange, Legend, Div, DateRangeSlider,
                          CustomJS, HoverTool, Title)
from bokeh.layouts import column, gridplot, row
from bokeh.palettes import RdYlBu11
from bokeh.transform import factor_cmap, factor_mark
from bokeh.io import curdoc

output_notebook()

# Load dataset
df = pd.read_csv('datasets/daily_min_temperatures.csv', parse_dates=['Date'])
df = df.dropna(subset=['Temp'])  # Drop any NaN (no '?' found)
df['Temperature'] = df['Temp'].astype(float)
df.drop('Temp', axis=1, inplace=True)

print(f"Dataset shape: {df.shape}")
print(df.head())
print(df.info())

## Question 1: Basic Time Series Line Chart

In [ ]:
# Q1: Basic line chart
source = ColumnDataSource(df)

p1 = figure(title="Daily Minimum Temperatures", 
           x_axis_label="Date", 
           y_axis_label="Temperature (°C)",
           height=400, width=800,
           tools='pan,wheel_zoom,box_zoom,reset,save')

hover1 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Temperature", "@Temperature{0.1f}°C")],
                  formatters={'@Date': 'datetime'})
p1.add_tools(hover1)

p1.line('Date', 'Temperature', source=source, line_width=2, line_color='blue')

show(p1)

## Question 2: 30-Day Rolling Average

In [ ]:
# Q2: Rolling average
df['Rolling_Avg'] = df['Temperature'].rolling(window=30, center=True).mean()
source2 = ColumnDataSource(df)

p2 = figure(title="Temperature with 30-Day Rolling Average", 
           x_axis_label="Date", y_axis_label="Temperature (°C)",
           height=400, width=800,
           tools='pan,wheel_zoom,reset')

hover2 = HoverTool(tooltips=[
    ("Date", "@Date{%F}"), 
    ("Temperature", "@Temperature{0.1f}°C"),
    ("Rolling Avg", "@Rolling_Avg{0.1f}°C")
], formatters={'@Date': 'datetime'})
p2.add_tools(hover2)

# Original temperature (solid blue)
r1 = p2.line('Date', 'Temperature', source=source2, line_width=1, line_color='blue', 
             legend_label='Daily Temperature', legend_line_width=2)

# Rolling average (dashed red)
r2 = p2.line('Date', 'Rolling_Avg', source=source2, line_width=2, line_color='red',
             line_dash='dashed', legend_label='30-Day Rolling Avg')

p2.legend.location = "top_left"
p2.legend.click_policy="hide"

show(p2)

## Question 3: Monthly Box Plots

In [ ]:
# Q3: Monthly box plots
df['Month'] = df['Date'].dt.month_name()
monthly_stats = df.groupby('Month')['Temperature'].agg(['min', 'max', 'median', 
                                                         lambda x: x.quantile(0.25), 
                                                         lambda x: x.quantile(0.75)]).round(1)
monthly_stats.columns = ['min', 'max', 'median', 'q1', 'q3']
monthly_stats = monthly_stats.reindex(['January', 'February', 'March', 'April', 'May', 'June',
                                       'July', 'August', 'September', 'October', 'November', 'December'])
source_monthly = ColumnDataSource(monthly_stats.reset_index())

p3 = figure(title="Monthly Temperature Box Plots", 
           x_range=list(monthly_stats.index), y_axis_label="Temperature (°C)",
           height=500, width=800, toolbar_location='right')

# Whiskers
lower_whisker = ColumnDataSource(monthly_stats[['Month', 'min']])
upper_whisker = ColumnDataSource(monthly_stats[['Month', 'max']])

p3.segment('Month', 'min', 'Month', 'q1', source=source_monthly, line_color='black')
p3.segment('Month', 'q3', 'Month', 'max', source=source_monthly, line_color='black')

# Boxes
p3.vbar('Month', 0.7, 'q1', 'q3', source=source_monthly, fill_color='navy', line_color='black')

# Medians
p3.segment('Month', 'median', 'Month', 'median', source=source_monthly,
          color='white', line_width=3)

hover3 = HoverTool(tooltips=[
    ("Month", "@Month"),
    ("Min", "@min"),( "Max", "@max"),
    ("Median", "@median")
])
p3.add_tools(hover3)

p3.xgrid.grid_line_color = None
show(p3)

## Question 4: Annual Box Plots with Color Mapping

In [ ]:
# Q4: Annual box plots with color by median
df['Year'] = df['Date'].dt.year
annual_stats = df.groupby('Year')['Temperature'].agg(['min', 'max', 'median', 'quantile'])
annual_stats['q1'] = annual_stats['quantile'].apply(lambda x: np.percentile(x, 25))
annual_stats['q3'] = annual_stats['quantile'].apply(lambda x: np.percentile(x, 75))
annual_stats.drop('quantile', axis=1, inplace=True)
annual_stats = annual_stats.round(1)
source_annual = ColumnDataSource(annual_stats.reset_index())

years = list(annual_stats.index.astype(str))
median_cmap = factor_cmap('Year_str', RdYlBu11, years)

p4 = figure(title="Annual Temperature Box Plots (Colored by Median)",
           x_range=years, y_axis_label="Temperature (°C)",
           height=500, width=1000, toolbar_location='right')

# Boxes colored by median
p4.vbar('Year_str', 0.8, 'q1', 'q3', source=source_annual,
        fill_color=median_cmap, line_color='black')

# Whiskers
p4.segment('Year_str', 'min', 'Year_str', 'q1', source=source_annual, line_color='black')
p4.segment('Year_str', 'q3', 'Year_str', 'max', source=source_annual, line_color='black')

# Medians
p4.segment('Year_str', 'median', 'Year_str', 'median', source=source_annual,
          color='white', line_width=2)

hover4 = HoverTool(tooltips=[
    ("Year", "@Year"),
    ("Min", "@min"),( "Q1", "@q1"),( "Median", "@median"),( "Q3", "@q3"),( "Max", "@max")
])
p4.add_tools(hover4)

p4.xgrid.grid_line_color = None
show(p4)

## Question 5: Interactive Date Range Slider

In [ ]:
# Q5: Interactive date range slider
source_slider = ColumnDataSource(df)

p5 = figure(title="Interactive Temperature Chart with Date Slider",
           x_axis_label="Date", y_axis_label="Temperature (°C)",
           height=400, width=800,
           tools='pan,wheel_zoom,reset')

r5 = p5.line('Date', 'Temperature', source=source_slider, line_width=2, line_color='green')

hover5 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Temp", "@Temperature{0.1f}°C")],
                  formatters={'@Date': 'datetime'})
p5.add_tools(hover5)

# Date range slider
date_slider = DateRangeSlider(title="Date Range",
                             start=df['Date'].min(),
                             end=df['Date'].max(),
                             value=(df['Date'].min(), df['Date'].max()),
                             step=1)

# JavaScript callback
callback = CustomJS(args=dict(source=source_slider, slider=date_slider),
                   code="""
    const data = source.data;
    const min_date = slider.start;
    const max_date = slider.end;
    
    const dates = data['Date'];
    const temps = data['Temperature'];
    
    const filtered_dates = [];
    const filtered_temps = [];
    
    for (let i = 0; i < dates.length; i++) {
        if (dates[i] >= min_date && dates[i] <= max_date) {
            filtered_dates.push(dates[i]);
            filtered_temps.push(temps[i]);
        }
    }
    
    source.change.emit();
    source.data = {
        ...source.data,
        'Date': filtered_dates,
        'Temperature': filtered_temps
    };
""")

date_slider.js_on_change('value', callback)

layout_slider = column(p5, date_slider)
show(layout_slider)

## Question 6: Time Series Decomposition

In [ ]:
# Q6: Time series decomposition
monthly = df.set_index('Date')['Temperature'].resample('M').mean()
trend = monthly.rolling(window=12, center=True).mean()
seasonality = monthly - trend

# Prepare data for Bokeh
decomp_df = pd.DataFrame({
    'Date': monthly.index,
    'Monthly': monthly.values,
    'Trend': trend.values,
    'Seasonality': seasonality.values
}).reset_index(drop=True)

source_decomp = ColumnDataSource(decomp_df)

# Three subplots
p61 = figure(title="Monthly Data", height=300, width=900, x_axis_label='Date', y_axis_label='Temperature (°C)')
p61.line('Date', 'Monthly', source=source_decomp, line_color='blue', line_width=2)

p62 = figure(title="Trend (12-month MA)", height=300, width=900, x_axis_label='Date', y_axis_label='Temperature (°C)')
p62.line('Date', 'Trend', source=source_decomp, line_color='green', line_width=2)

p63 = figure(title="Seasonality", height=300, width=900, x_axis_label='Date', y_axis_label='Temperature (°C)')
p63.line('Date', 'Seasonality', source=source_decomp, line_color='red', line_width=2)

# Shared x-axis
p61.x_range = p62.x_range = p63.x_range

# Grid layout
decomp_plot = gridplot([[p61], [p62], [p63]], toolbar_location='right')

show(decomp_plot)

print("Decomposition complete!")
print(monthly.head())

## Summary

✅ **All Lab 7 requirements completed**:
- Setup: Data loaded, cleaned, parsed
- Q1: Basic Bokeh line chart with tools/tooltips
- Q2: 30-day rolling average dual plot
- Q3: Monthly box plots with glyphs
- Q4: Annual box plots colored by median
- Q5: Interactive DateRangeSlider with CustomJS
- Q6: Time series decomposition (monthly, trend, seasonality)

All plots use `output_notebook()` and include proper interactivity.